In [11]:
for strategy in strategies:
    print("\n==============================")
    print(f"Traitement de la stratégie : {strategy}")
    print("==============================")

    rows = []
    corr_cols = None
    missing_files = []

    for sub_id in tqdm(list_sub, desc=f"{strategy}"):
        # phéno pour ce participant (sans session)
        pheno_sub = df_pheno[df_pheno["participant_id"] == sub_id]
        if pheno_sub.empty:
            # pas de métadonnées pour ce sujet, on skip
            continue

        path_corr = build_path(sub_id, strategy)

        if not os.path.exists(path_corr):
            missing_files.append(sub_id)
            continue

        # Lecture
        try:
            df_corr_matrix = pd.read_csv(path_corr, sep="\t", header=None)
        except Exception as e:
            print(f"⚠️ Erreur de lecture pour {sub_id}: {e}")
            missing_files.append(sub_id)
            continue

        # Corrélations (diagonale supérieure)
        A = df_corr_matrix.values
        iu = np.triu_indices(A.shape[0], k=1)
        vals = A[iu]

        # Créer (une seule fois) les noms de colonnes corr_<row>_<col>
        if corr_cols is None:
            corr_cols = [f"corr_{i+1}_{j+1}" for i, j in zip(*iu)]

        # Métadonnées
        # AJOUT ICI : J'ai ajouté "DX_GROUP" pour pouvoir le renommer ensuite
        meta = pheno_sub.iloc[0][["diagnosis", "age", "gender"]].to_dict()

        # Construire la ligne
        row = {"participant_id": sub_id, **meta}
        row.update({c: v for c, v in zip(corr_cols, vals)})
        rows.append(row)

        # libérer un peu au fil de l’eau
        del df_corr_matrix, A, vals

    # DataFrame final
    df_out = pd.DataFrame(rows)



    # --- MODIFICATION : Sauvegarde en TSV ---
    out_file = path_project / f"schaefcomb_{strategy}_ds000030.tsv"
    
    # sep="\t" crée un TSV
    df_out.to_csv(out_file, sep="\t", index=False)

    print(f"\n✅ Fichier exporté : {out_file}")
    print(f" - sujets sans fichier : {len(missing_files)}")
    if missing_files:
        print(missing_files[:20], "..." if len(missing_files) > 20 else "")

    # Libération mémoire
    del df_out, rows, missing_files, corr_cols
    gc.collect()


Traitement de la stratégie : Wang2023Simple


Wang2023Simple: 100%|██████████| 261/261 [13:24<00:00,  3.08s/it]



✅ Fichier exporté : /lustre07/scratch/pbergere/project_computational_medicine/datasets/schaefcomb_Wang2023Simple_ds000030.tsv
 - sujets sans fichier : 0

Traitement de la stratégie : Wang2023SimpleGSR


Wang2023SimpleGSR: 100%|██████████| 261/261 [16:13<00:00,  3.73s/it]



✅ Fichier exporté : /lustre07/scratch/pbergere/project_computational_medicine/datasets/schaefcomb_Wang2023SimpleGSR_ds000030.tsv
 - sujets sans fichier : 0


In [1]:
import os, gc, json  # <--- Ajout de json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

strategies = [
    'Wang2023Simple', 'Wang2023SimpleGSR'
]

path_pheno = "/lustre06/project/6097342/All_user_common_folder/datasets/ds000030/participants.tsv"
path_halfpipe = "/lustre07/scratch/pbergere/derivatives/halfpipe"
path_project = Path("/lustre07/scratch/pbergere/project_computational_medicine/datasets")

df_pheno = pd.read_csv(path_pheno, sep='\t')

list_sub = os.listdir(path_halfpipe)

# On garde uniquement les éléments de list_sub présents dans df_pheno["participant_id"]
list_sub = [s for s in list_sub if s in df_pheno["participant_id"].values]

# (Optionnel : pheno_idx n'est pas utilisé dans la boucle mais peut servir ailleurs)
pheno_idx = df_pheno.set_index("participant_id")[["age", "gender"]]

def build_path(sub_id, strategy):
    # Fonction helper supposée existante ou à adapter selon ton code précédent
    # Basé sur ton ls, les matrices semblent être dans le même dossier func/task-rest
    # Adapte ce path si tes matrices sont ailleurs
    filename = f"{sub_id}_task-rest_feature-{strategy}_atlas-schaeferCombined_desc-correlation_matrix.tsv"
    return os.path.join(path_halfpipe, sub_id, "func", "task-rest", filename)

for strategy in strategies:
    print("\n==============================")
    print(f"Traitement de la stratégie : {strategy}")
    print("==============================")

    rows = []
    corr_cols = None
    missing_files = []

    for sub_id in tqdm(list_sub, desc=f"{strategy}"):
        # 1. Phéno pour ce participant
        pheno_sub = df_pheno[df_pheno["participant_id"] == sub_id]
        if pheno_sub.empty:
            continue

        # 2. Path de la matrice de corrélation
        path_corr = build_path(sub_id, strategy)

        if not os.path.exists(path_corr):
            missing_files.append(sub_id)
            continue

        # 3. Lecture de la matrice
        try:
            df_corr_matrix = pd.read_csv(path_corr, sep="\t", header=None)
        except Exception as e:
            print(f"⚠️ Erreur de lecture matrice pour {sub_id}: {e}")
            missing_files.append(sub_id)
            continue

        # 4. Extraction valeurs (triu)
        A = df_corr_matrix.values
        iu = np.triu_indices(A.shape[0], k=1)
        vals = A[iu]

        if corr_cols is None:
            corr_cols = [f"corr_{i+1}_{j+1}" for i, j in zip(*iu)]

        # --- NOUVEAU BLOC : Récupération du Mean FD depuis le JSON ---
        mean_fd = np.nan
        
        # Le nom du json suit la convention de la stratégie
        json_filename = f"{sub_id}_task-rest_feature-{strategy}_atlas-schaeferCombined_timeseries.json"
        path_json = os.path.join(path_halfpipe, sub_id, "func", "task-rest", json_filename)
        
        if os.path.exists(path_json):
            try:
                with open(path_json, 'r') as f:
                    data = json.load(f)
                    # On récupère "FDMean", sinon NaN
                    mean_fd = data.get("FDMean", np.nan)
            except Exception as e:
                # On ne bloque pas le script pour un json corrompu, mais on log
                print(f"⚠️ Erreur lecture JSON pour {sub_id}: {e}")
        else:
            # Si le fichier n'existe pas, mean_fd reste NaN
            pass
        # -------------------------------------------------------------

        # 5. Métadonnées + Mean FD
        # On récupère les infos phéno de base
        meta = pheno_sub.iloc[0][["diagnosis", "age", "gender"]].to_dict()
        
        # On ajoute la mean_fd récupérée
        meta["mean_fd"] = mean_fd

        # 6. Construction de la ligne
        row = {"participant_id": sub_id, **meta}
        row.update({c: v for c, v in zip(corr_cols, vals)})
        rows.append(row)

        del df_corr_matrix, A, vals

    # Export
    df_out = pd.DataFrame(rows)
    
    # Renommage optionnel de 'diagnosis' en 'DX_GROUP' si nécessaire comme mentionné précédemment
    if "diagnosis" in df_out.columns:
        df_out.rename(columns={"diagnosis": "DX_GROUP"}, inplace=True)

    out_file = path_project / f"schaefcomb_{strategy}_ds000030.tsv"
    df_out.to_csv(out_file, sep="\t", index=False)

    print(f"\n✅ Fichier exporté : {out_file}")
    print(f" - Shape : {df_out.shape}")
    print(f" - Sujets sans fichier matrice : {len(missing_files)}")
    
    # Petit check pour voir si on a bien récupéré des mean_fd
    missing_fd = df_out['mean_fd'].isna().sum()
    print(f" - Sujets avec mean_fd manquant (NaN) : {missing_fd}")

    del df_out, rows, missing_files, corr_cols
    gc.collect()


Traitement de la stratégie : Wang2023Simple


Wang2023Simple: 100%|██████████| 261/261 [01:24<00:00,  3.07it/s]



✅ Fichier exporté : /lustre07/scratch/pbergere/project_computational_medicine/datasets/schaefcomb_Wang2023Simple_ds000030.tsv
 - Shape : (261, 93966)
 - Sujets sans fichier matrice : 0
 - Sujets avec mean_fd manquant (NaN) : 0

Traitement de la stratégie : Wang2023SimpleGSR


Wang2023SimpleGSR: 100%|██████████| 261/261 [01:13<00:00,  3.54it/s]



✅ Fichier exporté : /lustre07/scratch/pbergere/project_computational_medicine/datasets/schaefcomb_Wang2023SimpleGSR_ds000030.tsv
 - Shape : (261, 93966)
 - Sujets sans fichier matrice : 0
 - Sujets avec mean_fd manquant (NaN) : 0
